In [ ]:
from jupyterlab_h5web import H5Web

path = "/home/kaiobach/Downloads/Agate_Quartz_AGH.h5oina"
path = "/home/kaiobach/Downloads/718_microVol_053.up2"
path = "/mnt/production/nexus_paper/300/Patterns.up2"
# H5Web(path)

In [ ]:
import kikuchipy as kp
# path = "/home/kaiobach/Downloads/17944277/3D_EBSD_CathodeParticle_RawPatterns_Slice 1/3D_EBSD_CathodeParticle_RawPatterns_Slice (1).up2"

In [ ]:
dat = kp.load(path, lazy=True)
dat

In [ ]:
print(dat.axes_manager)
print(f"{type(dat.data)}, {dat.data.dtype}")
import h5py

n_chunks = np.shape(dat)[0]
print(n_chunks)
with h5py.File("/mnt/production/nexus_paper/ElectronDiffraction.nxs", "w") as h5w:
    # https://zenodo.org/records/11403783
    # optimal overwriting the auto chunker for this dataset
    # 1 * 1024 ** 2 / (2 * 446 * 446)
    prefix = "stack_2d"  # "/entry1/measurement/event1/image1/stack_2d"
    grp = h5w.create_group(prefix)
    grp.attrs["NX_class"] = "NXdata"
    grp.attrs["signal"] = "intensity"
    grp.attrs["axes"] = ["indices_image", "axis_j", "axis_i"]
    grp.attrs["axis_i_indices"] = np.uint32(0)
    grp.attrs["axis_j_indices"] = np.uint32(1)
    grp.attrs["indices_image_indices"] = np.uint32(2)
    dst = h5w.create_dataset(
        f"{prefix}/title",
        data="Kikuchi pattern stack",
    )
    dst = h5w.create_dataset(
        f"{prefix}/intensity",
        compression="gzip",
        compression_opts=9,
        chunks=(2, 446, 446),
        data=dat.inav[0:4],
    )
    for dim_idx, dim_suffix in enumerate(["i", "j"]):
        dst = h5w.create_dataset(
            f"{prefix}/axis_{dim_suffix}",
            compression="gzip",
            compression_opts=9,
            chunks=(446,),
            data=np.asarray(np.linspace(0, 446 - 1, num=446, endpoint=True), np.uint32),
        )
    dst = h5w.create_dataset(
        f"{prefix}/indices_image",
        compression="gzip",
        compression_opts=9,
        chunks=(4,),
        data=np.asarray(np.linspace(0, 4 - 1, num=4, endpoint=True), np.uint32),
    )

In [ ]:
from jupyterlab_h5web import H5Web

H5Web("/mnt/production/nexus_paper/ElectronDiffraction.nxs")
# (129**2) * (258**2)*2/1024/1024/1024

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dat.inav[1].plot()